# Andrew Brain: Unsloth Fine-Tuning Kit
This script is designed to run in **Google Colab** on the Free T4 GPU.
It uses `unsloth` to fine-tune Llama-3.1-8B-bnb-4bit on your `andrew_dataset.jsonl`.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Andrew typically deals in short, punchy dialogues.
dtype = None # Auto detection
load_in_4bit = True # Ensures it fits on Free Tier.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

In [ ]:
# Load your synthetic dataset uploaded to colab!
from datasets import load_dataset
# Ensure you upload 'andrew_dataset.jsonl' to the runtime /content folder first.
dataset = load_dataset("json", data_files="andrew_dataset.jsonl", split="train")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import os

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/Andrew_Checkpoints", # Saving checkpoints to your Drive!
        save_steps=15, # Save every 15 steps to bypass Colab timeouts
    ),
)

trainer_stats = trainer.train()

In [ ]:
## Exporting the Model
model.save_pretrained("lora_andrew_brain")
tokenizer.save_pretrained("lora_andrew_brain")

## Merge to GGUF format for Ollama / Oracle Free Tier Execution
model.save_pretrained_gguf("/content/drive/MyDrive/Andrew_Brain_GGUF", tokenizer, quantization_method = "q4_k_m")
print("Model saved to your Google Drive! Ready to upload to Oracle.")